In [0]:
# ===========================================
# Notebook Name:
# 01_finalize_pipeline_task
#
# Purpose:
# Last task of every Workflow run
# (run_if: ALL_DONE). Closes out the
# pipeline_run_log "running" row started by
# 00_initialize_pipeline_run, recomputes
# what changed during the run by comparing
# table timestamps against the run's
# started_at, and MERGEs a summary into
# pipeline_state for the next run to compare
# against.
#
# Note:
# Individual ingest/silver/gold tasks are not
# instrumented with per-task taskValues, so
# this notebook measures the run's effect by
# querying each layer's own timestamp columns
# (scraped_at / updated_at / first_seen_at)
# rather than aggregating a value emitted by
# every upstream task.
#
# Input:
# pokemon.bronze.scrape_log
# pokemon.silver.tournaments
# pokemon.silver.tournament_results
# pokemon.silver.decks
#
# Output:
# pokemon.ops.pipeline_run_log
# pokemon.ops.pipeline_state
# ===========================================

In [0]:
import json

from pyspark.sql import functions as F

PIPELINE_RUN_LOG_TABLE = (
    "pokemon.ops.pipeline_run_log"
)
PIPELINE_STATE_TABLE = (
    "pokemon.ops.pipeline_state"
)
SCRAPE_LOG_TABLE = "pokemon.bronze.scrape_log"
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"
TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)
DECKS_TABLE = "pokemon.silver.decks"

ERROR_STATUS_VALUES = [
    "error",
    "failed",
    "failure",
    "http_error",
]

pipeline_run_id = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="pipeline_run_id",
    default=None,
    debugValue="manual-run",
)
pipeline_name = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="pipeline_name",
    default="daily_pokemon_lakehouse_pipeline",
    debugValue="daily_pokemon_lakehouse_pipeline",
)
started_at_iso = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="started_at_iso",
    default=None,
    debugValue=None,
)

print("pipeline_run_id:", pipeline_run_id)
print("pipeline_name  :", pipeline_name)
print("started_at_iso :", started_at_iso)

In [0]:
if (
    pipeline_run_id is None
    or started_at_iso is None
):
    fallback_row = (
        spark.table(PIPELINE_RUN_LOG_TABLE)
        .filter(
            F.col("task_name") == "pipeline"
        )
        .filter(
            F.col("status") == "running"
        )
        .orderBy(
            F.col("started_at").desc()
        )
        .limit(1)
        .first()
    )

    if fallback_row is None:
        raise ValueError(
            "No pipeline_run_id available from "
            "taskValues and no 'running' row "
            f"found in {PIPELINE_RUN_LOG_TABLE}. "
            "Run 00_initialize_pipeline_run first."
        )

    pipeline_run_id = (
        fallback_row["pipeline_run_id"]
    )
    started_at_iso = (
        fallback_row["started_at"].isoformat()
    )

    print(
        "Recovered pipeline_run_id from "
        f"{PIPELINE_RUN_LOG_TABLE}: "
        f"{pipeline_run_id}"
    )

started_at = F.to_timestamp(
    F.lit(started_at_iso)
)

In [0]:
def count_since(table_name, timestamp_column):
    return (
        spark.table(table_name)
        .filter(
            F.col(timestamp_column) >= started_at
        )
        .count()
    )


new_tournament_count = count_since(
    TOURNAMENTS_TABLE, "first_seen_at"
)
new_result_count = count_since(
    TOURNAMENT_RESULTS_TABLE, "updated_at"
)
new_deck_count = count_since(
    DECKS_TABLE, "updated_at"
)

error_df = (
    spark.table(SCRAPE_LOG_TABLE)
    .filter(
        F.col("scraped_at") >= started_at
    )
    .filter(
        F.lower(F.col("status")).isin(
            *ERROR_STATUS_VALUES
        )
    )
)

error_count = error_df.count()

print(
    "New/changed tournaments:",
    new_tournament_count,
)
print(
    "New/changed tournament_results:",
    new_result_count,
)
print(
    "New/changed decks:",
    new_deck_count,
)
print("Scrape errors:", error_count)

if error_count > 0:
    display(
        error_df.select(
            "source_type",
            "source_id",
            "status",
            "error_message",
            "scraped_at",
        )
    )

In [0]:
run_status = (
    "failed" if error_count > 0 else "success"
)

finished_row_df = spark.createDataFrame(
    [{
        "pipeline_run_id": pipeline_run_id,
        "task_name": "pipeline",
        "status": run_status,
        "input_count": (
            new_tournament_count
            + new_result_count
            + new_deck_count
        ),
        "insert_count": new_deck_count,
        "update_count": (
            new_tournament_count
            + new_result_count
        ),
        "skip_count": 0,
        "error_count": error_count,
        "error_message": (
            None
            if error_count == 0
            else (
                f"{error_count} scrape_log "
                "error rows since run start"
            )
        ),
    }],
    schema="""
    pipeline_run_id STRING,
    task_name STRING,
    status STRING,
    input_count BIGINT,
    insert_count BIGINT,
    update_count BIGINT,
    skip_count BIGINT,
    error_count BIGINT,
    error_message STRING
    """,
).withColumn(
    "finished_at", F.current_timestamp()
)

finished_row_df.createOrReplaceTempView(
    "staged_finalize_run_log"
)

spark.sql(f"""
MERGE INTO {PIPELINE_RUN_LOG_TABLE} AS target
USING staged_finalize_run_log AS source
ON  target.pipeline_run_id = source.pipeline_run_id
AND target.task_name = source.task_name

WHEN MATCHED THEN UPDATE SET
    target.status = source.status,
    target.finished_at = source.finished_at,
    target.elapsed_ms =
        CAST(
            (
                unix_timestamp(source.finished_at)
                - unix_timestamp(target.started_at)
            ) * 1000 AS BIGINT
        ),
    target.input_count = source.input_count,
    target.insert_count = source.insert_count,
    target.update_count = source.update_count,
    target.skip_count = source.skip_count,
    target.error_count = source.error_count,
    target.error_message = source.error_message
""")

print(
    "pipeline_run_log: recorded "
    f"'{run_status}' for "
    f"pipeline_run_id={pipeline_run_id}"
)

In [0]:
if spark.catalog.tableExists(
    PIPELINE_STATE_TABLE
):
    previous_failure_count_row = (
        spark.table(PIPELINE_STATE_TABLE)
        .filter(
            (F.col("pipeline_name") == pipeline_name)
            & (F.col("stage_name") == "overall")
        )
        .select("consecutive_failure_count")
        .first()
    )

else:
    previous_failure_count_row = None

previous_failure_count = (
    previous_failure_count_row[
        "consecutive_failure_count"
    ]
    if previous_failure_count_row is not None
    else 0
)

next_failure_count = (
    0
    if run_status == "success"
    else previous_failure_count + 1
)

metrics = {
    "new_tournament_count": new_tournament_count,
    "new_result_count": new_result_count,
    "new_deck_count": new_deck_count,
    "error_count": error_count,
}

state_df = spark.createDataFrame(
    [{
        "pipeline_name": pipeline_name,
        "stage_name": "overall",
        "last_run_status": run_status,
        "watermark_value": started_at_iso,
        "metrics_json": json.dumps(metrics),
        "consecutive_failure_count": (
            next_failure_count
        ),
    }],
    schema="""
    pipeline_name STRING,
    stage_name STRING,
    last_run_status STRING,
    watermark_value STRING,
    metrics_json STRING,
    consecutive_failure_count INT
    """,
).withColumn(
    "last_run_at", F.current_timestamp()
).withColumn(
    "last_success_at",
    F.current_timestamp()
    if run_status == "success"
    else F.lit(None).cast("timestamp"),
)

state_df.createOrReplaceTempView(
    "staged_pipeline_state"
)

spark.sql(f"""
MERGE INTO {PIPELINE_STATE_TABLE} AS target
USING staged_pipeline_state AS source
ON  target.pipeline_name = source.pipeline_name
AND target.stage_name = source.stage_name

WHEN MATCHED THEN UPDATE SET
    target.last_run_at = source.last_run_at,
    target.last_run_status = source.last_run_status,
    target.last_success_at = COALESCE(
        source.last_success_at,
        target.last_success_at
    ),
    target.watermark_value = source.watermark_value,
    target.metrics_json = source.metrics_json,
    target.consecutive_failure_count =
        source.consecutive_failure_count

WHEN NOT MATCHED THEN INSERT (
    pipeline_name,
    stage_name,
    last_run_at,
    last_run_status,
    last_success_at,
    watermark_value,
    metrics_json,
    consecutive_failure_count
)
VALUES (
    source.pipeline_name,
    source.stage_name,
    source.last_run_at,
    source.last_run_status,
    source.last_success_at,
    source.watermark_value,
    source.metrics_json,
    source.consecutive_failure_count
)
""")

print(
    "pipeline_state: recorded "
    f"pipeline_name={pipeline_name}, "
    f"stage_name=overall, "
    f"status={run_status}, "
    f"consecutive_failure_count="
    f"{next_failure_count}"
)

In [0]:
display(
    spark.table(PIPELINE_RUN_LOG_TABLE)
    .filter(
        F.col("pipeline_run_id") == pipeline_run_id
    )
)

display(
    spark.table(PIPELINE_STATE_TABLE)
    .filter(F.col("pipeline_name") == pipeline_name)
)